In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import tqdm
import os
import math
import matplotlib.pyplot as plt
from IPython.display import clear_output

In [ ]:
# Set up device and optimizations
torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Diffusion schedules
T = 1000
beta_start, beta_end = 1e-4, 0.02
betas = torch.linspace(beta_start, beta_end, T).to(device)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, dim=0).to(device)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod).to(device)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1 - alphas_cumprod).to(device)

In [ ]:
# Simple Digit classifier to evaluate accuracy (using a small MLP)
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.fc(x)

In [ ]:
import math

# Guided UNet model with label conditioning
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
        if emb.shape[1] < self.dim:
            emb = F.pad(emb, (0, self.dim - emb.shape[1]))
        return emb

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
    def forward(self, x):
        identity = x
        x = F.relu(self.conv1(x))
        x = self.conv2(x)
        x += self.shortcut(identity)
        return F.relu(x)

class UNet(nn.Module):
    def __init__(self, time_dim=256, label_dim=10):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim)
        )
        self.label_mlp = nn.Sequential(
            nn.Linear(label_dim, time_dim),  # Map 10-class one-hot to time_dim
            nn.ReLU()
        )
        self.enc1 = ResidualBlock(1, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.enc3 = ResidualBlock(128, 256)
        self.time_proj1 = nn.Linear(time_dim, 64)
        self.time_proj2 = nn.Linear(time_dim, 128)
        self.time_proj3 = nn.Linear(time_dim, 256)
        self.bottleneck = ResidualBlock(256, 256)
        self.dec1 = ResidualBlock(512, 128)  # 256 + 256 from skip
        self.dec2 = ResidualBlock(256, 64)   # 128 + 128 from skip
        self.dec3 = nn.Conv2d(128, 1, 3, padding=1)  # 64 + 64 from skip

    def forward(self, x, t, labels):
        t_emb = self.time_mlp(t)
        l_emb = self.label_mlp(labels)  # (batch, 10) -> (batch, time_dim)
        combined_emb = t_emb + l_emb
        x1 = self.enc1(x) + self.time_proj1(combined_emb).unsqueeze(-1).unsqueeze(-1)
        x2 = self.enc2(x1) + self.time_proj2(combined_emb).unsqueeze(-1).unsqueeze(-1)
        x3 = self.enc3(x2) + self.time_proj3(combined_emb).unsqueeze(-1).unsqueeze(-1)
        x = self.bottleneck(x3)
        x = self.dec1(torch.cat([x, x3], dim=1))
        x = self.dec2(torch.cat([x, x2], dim=1))
        x = self.dec3(torch.cat([x, x1], dim=1))
        return x

In [ ]:
import torch

# Generate Digit Image from User Input
checkpoint_path_diffusion = "checkpoint_diffusion.pth"
print("Loading diffusion model checkpoint...")
checkpoint = torch.load(checkpoint_path_diffusion)
model = UNet().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Diffusion model loaded successfully.")

# Load pre-trained classifier
print("Loading classifier checkpoint...")
checkpoint_classifier = torch.load("checkpoint_classifier.pth")
classifier = DigitClassifier().to(device)
classifier.load_state_dict(checkpoint_classifier['classifier_state_dict'])
classifier.eval()
print("Classifier loaded successfully.")

@torch.no_grad()
def sample_digit(model, digit, img_size=28, channels=1):
    print(f"Generating image for digit {digit}...")
    label = torch.tensor([digit], device=device)
    labels_one_hot = F.one_hot(label, num_classes=10).float()
    x = torch.randn((1, channels, img_size, img_size)).to(device)  # Single sample
    for t in reversed(range(1, T)):
        if t % 100 == 0:
            clear_output(wait=True)
            print(f"  Denoising step {t}...")

            temp = torch.clamp(x, -1, 1)
            temp = temp * 0.5 + 0.5

            plt.figure(figsize=(4, 4))
            plt.imshow(temp.cpu().numpy().squeeze(), cmap='gray')
            plt.axis('off')
            plt.show()


        t_batch = torch.full((1,), t, device=device, dtype=torch.long)
        beta_t = betas[t].to(device)
        sqrt_one_minus_alphas_cumprod_t = sqrt_one_minus_alphas_cumprod[t].to(device)
        sqrt_recip_alpha_t = (1.0 / torch.sqrt(alphas[t])).to(device)
        epsilon_theta = model(x, t_batch.float(), labels_one_hot)
        model_mean = sqrt_recip_alpha_t * (x - beta_t * epsilon_theta / sqrt_one_minus_alphas_cumprod_t)

        if t > 1:
            noise = torch.randn_like(x).to(device)
            sigma_t = torch.sqrt(beta_t)
            x = model_mean + sigma_t * noise
        else:
            x = model_mean
    x = torch.clamp(x, -1, 1)
    print(f"Sampling complete. Output shape: {x.shape}")
    return  x * 0.5 + 0.5  # Unnormalize to [0, 1]

# Get user input and generate
while True:
    try:
        user_input = int(input("Enter a digit (0-9) to generate, or -1 to exit: "))
        if user_input == -1:
            print("Exiting generator. Goodbye! 🚀")
            break
        elif 0 <= user_input <= 9:
            sample = sample_digit(model, user_input)
            flat_sample = sample.view(1, -1)
            with torch.no_grad():
                pred_labels = classifier(flat_sample)
                _, predicted = torch.max(pred_labels, 1)
                print(f"Predicted label: {predicted.item()}")

            clear_output(wait=True)
            plt.figure(figsize=(4, 4))
            plt.imshow(sample[0].cpu().numpy().squeeze(), cmap='gray')
            plt.title(f"Generated Digit: {user_input}\nPredicted: {predicted.item()}")
            plt.axis('off')
            plt.show()
        else:
            print("Please enter a digit between 0 and 9, or -1 to exit.")
    except ValueError:
        print("Invalid input! Please enter a number between 0 and 9, or -1 to exit.")

    finally:
        clear_output(wait=True)